# Extracting Fact and Dimension Scripts

Please run the scripts in the following order: 1.extracting_script -> 2.geo_handle -> 3.association_mining

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Import Dataset

In [2]:
#Import the data set, skipping first 4 rows and using columns A to W
fatalities = pd.read_excel('bitre_fatalities_dec2024.xlsx', sheet_name='BITRE_Fatality', skiprows = 4, nrows=56875,  usecols = 'A:W')
fatalities.tail()

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Bus Involvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,...,Age,National Remoteness Areas,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Age Group,Day of week,Time of day
56869,19896006,Tas,1,1989,Wednesday,20:20:00,Multiple,No,-9,Yes,...,13,Unknown,NaN,NaN,Undetermined,No,No,0_to_16,Weekday,Night
56870,19896006,Tas,1,1989,Wednesday,20:20:00,Multiple,No,-9,Yes,...,13,Unknown,NaN,NaN,Undetermined,No,No,0_to_16,Weekday,Night
56871,19896006,Tas,1,1989,Wednesday,20:20:00,Multiple,No,-9,Yes,...,18,Unknown,NaN,NaN,Undetermined,No,No,17_to_25,Weekday,Night
56872,19896006,Tas,1,1989,Wednesday,20:20:00,Multiple,No,-9,Yes,...,14,Unknown,NaN,NaN,Undetermined,No,No,0_to_16,Weekday,Night
56873,19895133,WA,1,1989,Wednesday,21:00:00,Multiple,No,-9,No,...,70,Unknown,NaN,NaN,Undetermined,No,No,65_to_74,Weekday,Night


In [3]:
fatalities.replace([-9, '-9','Unknown','Undetermined'], np.nan, inplace=True)
fatalities.head()

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Bus Involvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,...,Age,National Remoteness Areas,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Age Group,Day of week,Time of day
0,20241115,NSW,12,2024,Friday,04:00:00,Single,No,No,No,...,74.0,Inner Regional Australia,Riverina,Wagga Wagga,Arterial Road,Yes,No,65_to_74,Weekday,Night
1,20241125,NSW,12,2024,Friday,06:15:00,Single,No,No,No,...,19.0,Inner Regional Australia,Sydney - Baulkham Hills and Hawkesbury,Hawkesbury,Local Road,No,No,17_to_25,Weekday,Day
2,20246013,Tas,12,2024,Friday,09:43:00,Multiple,No,No,No,...,33.0,Inner Regional Australia,Launceston and North East,Northern Midlands,Local Road,Yes,No,26_to_39,Weekday,Day
3,20241002,NSW,12,2024,Friday,10:35:00,Multiple,No,No,No,...,32.0,Outer Regional Australia,New England and North West,Armidale Regional,National or State Highway,No,No,26_to_39,Weekday,Day
4,20242261,Vic,12,2024,Friday,11:30:00,Multiple,NaN,NaN,NaN,...,62.0,NaN,NaN,NaN,NaN,No,No,40_to_64,Weekday,Day


In [4]:
fatalities.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56874 entries, 0 to 56873
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Crash ID                       56874 non-null  int64  
 1   State                          56874 non-null  object 
 2   Month                          56874 non-null  int64  
 3   Year                           56874 non-null  int64  
 4   Dayweek                        56874 non-null  object 
 5   Time                           56831 non-null  object 
 6   Crash Type                     56874 non-null  object 
 7   Bus Involvement                56806 non-null  object 
 8   Heavy Rigid Truck Involvement  36317 non-null  object 
 9   Articulated Truck Involvement  56812 non-null  object 
 10  Speed Limit                    55389 non-null  object 
 11  Road User                      56863 non-null  object 
 12  Gender                         56840 non-null 

In [5]:
#Import the data set, skipping first 4 rows and using columns A to T
crashes = pd.read_excel('bitre_fatal_crashes_dec2024.xlsx', sheet_name='BITRE_Fatal_Crash', skiprows = 4, nrows=51285,  usecols = 'A:T')
crashes.tail()

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Number Fatalities,Bus \nInvolvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,Speed Limit,National Remoteness Areas,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Day of week,Time of Day
51279,19891246,NSW,1,1989,Wednesday,17:05:00,Single,1,Yes,-9,No,60,Unknown,NaN,NaN,Undetermined,No,No,Weekday,Day
51280,19892038,Vic,1,1989,Wednesday,18:50:00,Single,1,Yes,No,No,60,Unknown,NaN,NaN,Undetermined,No,No,Weekday,Night
51281,19894064,SA,1,1989,Wednesday,19:00:00,Multiple,1,No,-9,No,100,Unknown,NaN,NaN,Undetermined,No,No,Weekday,Night
51282,19896006,Tas,1,1989,Wednesday,20:20:00,Multiple,6,No,-9,Yes,100,Unknown,NaN,NaN,Undetermined,No,No,Weekday,Night
51283,19895133,WA,1,1989,Wednesday,21:00:00,Multiple,1,No,-9,No,-9,Unknown,NaN,NaN,Undetermined,No,No,Weekday,Night


In [6]:
crashes.replace([-9, '-9','Unknown','Undetermined'], np.nan, inplace=True)

In [7]:
assert crashes.shape[0] == crashes['Crash ID'].nunique() #The number of rows in the crashes data set should be equal to the number of unique crash IDs

In [8]:
assert fatalities['Crash ID'].nunique() == crashes['Crash ID'].nunique() #The number of unique crash IDs in both data sets should be the same

In [9]:
crashes = crashes.rename(columns=lambda x: x.replace("\n", " ").strip()) #Remove the new line character in the crash data
crashes = crashes.rename(columns = {'Time of Day': 'Time of day',
                                    'Bus  Involvement': 'Bus Involvement',}) #Normalise the column name to be exact the same to fatality data
crashes_check = crashes.drop(columns = ['Number Fatalities','Time of day']).sort_values(by='Crash ID')
fatal_crash_check = fatalities[crashes_check.columns].drop_duplicates().sort_values(by='Crash ID')
crashes_check = crashes_check.reset_index(drop=True)
fatal_crash_check = fatal_crash_check.reset_index(drop=True)

In [10]:
#Check if the 2 datasets are in the same shape
print(crashes_check.shape, fatal_crash_check.shape)
print(crashes_check.columns.tolist())
print(fatal_crash_check.columns.tolist())

(51284, 18) (51285, 18)
['Crash ID', 'State', 'Month', 'Year', 'Dayweek', 'Time', 'Crash Type', 'Bus Involvement', 'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement', 'Speed Limit', 'National Remoteness Areas', 'SA4 Name 2021', 'National LGA Name 2021', 'National Road Type', 'Christmas Period', 'Easter Period', 'Day of week']
['Crash ID', 'State', 'Month', 'Year', 'Dayweek', 'Time', 'Crash Type', 'Bus Involvement', 'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement', 'Speed Limit', 'National Remoteness Areas', 'SA4 Name 2021', 'National LGA Name 2021', 'National Road Type', 'Christmas Period', 'Easter Period', 'Day of week']


In [11]:
print(fatal_crash_check['Crash ID'].duplicated().sum())  # Count duplicate rows

1


In [12]:
duplicate_rows = fatal_crash_check[fatal_crash_check.duplicated(subset=['Crash ID'], keep=False)] #Looking for any duplicated rows
duplicate_rows 

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Bus Involvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,Speed Limit,National Remoteness Areas,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Day of week
42203,20164024,SA,3,2016,Thursday,08:15:00,Single,No,No,No,60,Major Cities of Australia,Adelaide - North,Salisbury,Sub-arterial Road,No,No,Weekday
42204,20164024,SA,3,2016,Thursday,08:15:00,Single,No,No,No,100,Major Cities of Australia,Adelaide - North,Salisbury,Sub-arterial Road,No,No,Weekday


In [13]:
crashes_check[crashes_check['Crash ID'] == 20164024] # Check the crash ID in the crashes data set

,Crash ID,State,Month,Year,Dayweek,Time,Crash Type,Bus Involvement,Heavy Rigid Truck Involvement,Articulated Truck Involvement,Speed Limit,National Remoteness Areas,SA4 Name 2021,National LGA Name 2021,National Road Type,Christmas Period,Easter Period,Day of week
42203,20164024,SA,3,2016,Thursday,08:15:00,Single,No,No,No,60,Major Cities of Australia,Adelaide - North,Salisbury,Sub-arterial Road,No,No,Weekday


In [14]:
fatalities.loc[(fatalities['Crash ID'] == 20164024) & (fatalities['Speed Limit']==100),'Speed Limit'] = 60 #Convert the different speed to the same with crash data

In [15]:
#Check if the 2 datasets are in the same shape
crashes_check = crashes.drop(columns = ['Number Fatalities','Time of day']).sort_values(by='Crash ID')
fatal_crash_check = fatalities[crashes_check.columns].drop_duplicates().sort_values(by='Crash ID')
print(crashes_check.shape, fatal_crash_check.shape)
print(crashes_check.columns.tolist())
print(fatal_crash_check.columns.tolist())

(51284, 18) (51284, 18)
['Crash ID', 'State', 'Month', 'Year', 'Dayweek', 'Time', 'Crash Type', 'Bus Involvement', 'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement', 'Speed Limit', 'National Remoteness Areas', 'SA4 Name 2021', 'National LGA Name 2021', 'National Road Type', 'Christmas Period', 'Easter Period', 'Day of week']
['Crash ID', 'State', 'Month', 'Year', 'Dayweek', 'Time', 'Crash Type', 'Bus Involvement', 'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement', 'Speed Limit', 'National Remoteness Areas', 'SA4 Name 2021', 'National LGA Name 2021', 'National Road Type', 'Christmas Period', 'Easter Period', 'Day of week']


In [16]:
#See any remaining different in the 2 files
crashes_check = crashes_check.reset_index(drop=True)
fatal_crash_check = fatal_crash_check.reset_index(drop=True)
diff = crashes_check.compare(fatal_crash_check)
print(diff)

              National Road Type      
                            self other
41667  National or State Highway   NaN
41668          Sub-arterial Road   NaN
41669             Collector Road   NaN
41670              Arterial Road   NaN
41671              Arterial Road   NaN
...                          ...   ...
41937              Arterial Road   NaN
41938              Arterial Road   NaN
41939          Sub-arterial Road   NaN
41940              Arterial Road   NaN
41941                Access road   NaN

[275 rows x 2 columns]


In [17]:
diff.info()

<class 'pandas.core.frame.DataFrame'>
Index: 275 entries, 41667 to 41941
Data columns (total 2 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   (National Road Type, self)   275 non-null    object
 1   (National Road Type, other)  0 non-null      object
dtypes: object(2)
memory usage: 6.4+ KB


There are only one column different that is National Road Type.

As we find the fatality dataset is at higher granular level so we will consolidate fatality National Road Type data to setup for our ETL and analysis. Additionally, the remaining dimensions are perfectly aligned so we will not need any further action at this stage

In [18]:
roadtype_crash = crashes[['Crash ID', 'National Road Type']].drop_duplicates()
fatalities_temp = pd.merge(fatalities,
                           roadtype_crash,
                           on='Crash ID',
                           how = 'left',
                           suffixes=('','_crashes'))
fatalities_temp['National Road Type'] = fatalities_temp['National Road Type'].fillna(fatalities_temp['National Road Type_crashes'])
# Drop the temporary '_crashes' column after filling
fatalities_temp = fatalities_temp.drop(columns=['National Road Type_crashes'])
assert fatalities.shape[0] == fatalities_temp.shape[0]
fatalities = fatalities_temp.copy()

# Data Cleaning

Number of rows can be removed without over cross the thresh hold

In [19]:
threshold = 0.05 * len(fatalities)
threshold

2843.7000000000003

In [20]:
fatalities.isna().sum() #checking for missing values

Crash ID                             0
State                                0
Month                                0
Year                                 0
Dayweek                              0
Time                                43
Crash Type                           0
Bus Involvement                     68
Heavy Rigid Truck Involvement    20557
Articulated Truck Involvement       62
Speed Limit                       1485
Road User                           11
Gender                              34
Age                                116
National Remoteness Areas        45520
SA4 Name 2021                    45851
National LGA Name 2021           45849
National Road Type               45846
Christmas Period                     0
Easter Period                        0
Age Group                          117
Day of week                         13
Time of day                         44
dtype: int64

Checking if the number of rows that:

 - Missing all of the big vehicle involvement
 - Missing any of time, speed limit, road user, gender or age feature

In [21]:
fatalities_drop = fatalities[
    fatalities[['Bus Involvement', 'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement']].isna().all(axis=1) | #Remove rows where all heavy vehicle involement was not defined
    fatalities[['Time','Speed Limit','Road User', 'Gender', 'Age']].isna().any(axis=1) #Remove rows where any of the columns are not defined
]
assert len(fatalities_drop) < threshold #checking that the number of rows dropped is less than the threshold
len(fatalities_drop) #checking the number of rows dropped

1681

In [22]:
fatalities_clean = fatalities.drop(fatalities_drop.index)
fatalities_clean.isna().sum()

Crash ID                             0
State                                0
Month                                0
Year                                 0
Dayweek                              0
Time                                 0
Crash Type                           0
Bus Involvement                     10
Heavy Rigid Truck Involvement    19604
Articulated Truck Involvement        4
Speed Limit                          0
Road User                            0
Gender                               0
Age                                  0
National Remoteness Areas        43993
SA4 Name 2021                    44314
National LGA Name 2021           44312
National Road Type               44310
Christmas Period                     0
Easter Period                        0
Age Group                            1
Day of week                          0
Time of day                          1
dtype: int64

There is still 14 rows left missing Bus & Articulated Truck Involment that can be dropped with out crossing the threshold

In [23]:
fatalities_clean = fatalities_clean.dropna(subset=['Bus Involvement','Articulated Truck Involvement']) #Remove rows where bus or articulated truck involvement is not defined

In [24]:
fatalities_clean['National LGA Name 2021'] = fatalities_clean['National LGA Name 2021'].fillna(value = 'Unidentified_' + fatalities_clean['State'])
fatalities_clean['National Road Type'] =fatalities_clean['National Road Type'].fillna('Unknown_roadtype')
fatalities_clean['Heavy Rigid Truck Involvement'] = fatalities_clean['Heavy Rigid Truck Involvement'].fillna('Unknown_Involvement')

In [25]:
fatalities.shape[0] - fatalities_clean.shape[0]

1691

# Dimension Extraction

### Date Dimension

In [26]:
date_dimension = fatalities_clean[['Dayweek','Month','Year']].drop_duplicates()
date_dimension['DateKey'] = range(1, len(date_dimension)+1)
date_dimension = date_dimension[['DateKey','Dayweek','Month','Year']]
# date_dimension['Weekend'] = np.where(date_dimension['Dayweek'].isin(['Saturday', 'Sunday']), 'Y', 'N')
date_dimension.info()


<class 'pandas.core.frame.DataFrame'>
Index: 3024 entries, 0 to 56846
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   DateKey  3024 non-null   int64 
 1   Dayweek  3024 non-null   object
 2   Month    3024 non-null   int64 
 3   Year     3024 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 118.1+ KB


### Time Dimension

In [27]:
time_dimension = fatalities_clean[['Time']].drop_duplicates().sort_values(by='Time', ascending=True).reset_index(drop=True)
# time_dimension['TimeBand'] = pd.cut(pd.to_datetime(time_dimension['Time'], format='%H:%M:%S', errors='coerce').dt.hour, 
#                                      bins=[0,5,12,17,21,24], 
#                                      labels=['Night','Morning','Afternoon','Evening','Night'],
#                                      right= False,
#                                      ordered = False)
# time_dimension['RushHour'] = np.where(pd.to_datetime(time_dimension['Time'], format='%H:%M%S', errors ='coerce').dt.hour.isin([7,8,9,16,17,18]), 'Y', 'N')
time_dimension['TimeKey'] = range(1, len(time_dimension)+1)
time_dimension = time_dimension[['TimeKey','Time']]
time_dimension.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1419 entries, 0 to 1418
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   TimeKey  1419 non-null   int64 
 1   Time     1419 non-null   object
dtypes: int64(1), object(1)
memory usage: 22.3+ KB


In [28]:
time_dimension

,TimeKey,Time
0,1,00:00:00
1,2,00:01:00
2,3,00:02:00
3,4,00:03:00
4,5,00:04:00
...,...,...
1414,1415,23:55:00
1415,1416,23:56:00
1416,1417,23:57:00
1417,1418,23:58:00


### Road User Dimension

In [29]:
road_user_dimension = fatalities_clean[['Road User']].drop_duplicates().reset_index(drop=True)
road_user_dimension['RoadUserKey'] = range(1, len(road_user_dimension)+1)
road_user_dimension = road_user_dimension.rename(columns={'Road User': 'RoadUser'})
road_user_dimension = road_user_dimension[['RoadUserKey','RoadUser']]
road_user_dimension

,RoadUserKey,RoadUser
0,1,Driver
1,2,Passenger
2,3,Motorcycle rider
3,4,Pedestrian
4,5,Pedal cyclist
5,6,Motorcycle pillion passenger
6,7,Other/-9


### Gender Dimension

In [30]:
gender_dimension = fatalities_clean[['Gender']].drop_duplicates().reset_index(drop=True)
gender_dimension['GenderKey'] = range(1, len(gender_dimension)+1)
gender_dimension = gender_dimension[['GenderKey','Gender']]
gender_dimension

,GenderKey,Gender
0,1,Male
1,2,Female


In [31]:
age_dimension = fatalities_clean[['Age']].drop_duplicates().reset_index(drop=True)
age_dimension['Age'] = age_dimension['Age'].astype('int') #Convert age to numeric
# age_dimension['AgeBand'] = pd.cut(age_dimension['Age'],
#                                   bins=[0,16,21,30,50,65,150],
#                                   labels=['Under Age','Teenager','Young Adult','Midldle-aged Adult','Old Adult','Senior'],
#                                   right=False
#                                   )
# age_dimension = age_dimension[['Age','AgeBand']].sort_values(by='Age', ascending=True).reset_index(drop=True)   
age_dimension

,Age
0,74
1,19
2,33
3,32
4,61
...,...
97,10
98,99
99,95
100,98


### Speed Limit Dimension

In [32]:
fatalities_clean['Speed Limit']

0        100
1         80
2         50
3        100
5        100
        ... 
56868    100
56869    100
56870    100
56871    100
56872    100
Name: Speed Limit, Length: 55183, dtype: object

In [33]:
fatalities_clean['Speed Limit'].replace('<40',40, inplace=True)
fatalities_clean['Speed Limit'] = fatalities_clean['Speed Limit'].astype(int) #Convert the speed limit to int

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_6752\2913706305.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  fatalities_clean['Speed Limit'].replace('<40',40, inplace=True)


In [34]:
speed_dimension = fatalities_clean[['Speed Limit']].drop_duplicates().reset_index(drop=True)
speed_dimension = speed_dimension.rename(columns={'Speed Limit': 'SpeedLimit'})
# speed_dimension['SpeedBand'] = pd.cut(speed_dimension['SpeedLimit'],
#                                      bins=[0,50,100,1000],
#                                      labels=['Under50','From50to100','Over100'],
#                                      right=False
#                                      )
speed_dimension = speed_dimension.sort_values(by='SpeedLimit', ascending=True).reset_index(drop=True)
speed_dimension

,SpeedLimit
0,5
1,10
2,15
3,20
4,25
5,30
6,40
7,50
8,60
9,70


### Road Dimension

In [35]:
road_dimension = fatalities_clean[['National Road Type']].drop_duplicates().reset_index(drop=True)
road_dimension['RoadKey'] = range(1, len(road_dimension)+1)
road_dimension = road_dimension.rename(columns={'National Road Type': 'RoadType'})
road_dimension = road_dimension[['RoadKey','RoadType']]
road_dimension

,RoadKey,RoadType
0,1,Arterial Road
1,2,Local Road
2,3,National or State Highway
3,4,Sub-arterial Road
4,5,Unknown_roadtype
5,6,Collector Road
6,7,Pedestrian Thoroughfare
7,8,Access road
8,9,Busway


### Involvement Dimension

In [36]:
involvement_dimension = fatalities_clean[['Bus Involvement', 'Heavy Rigid Truck Involvement', 'Articulated Truck Involvement']].drop_duplicates().reset_index(drop=True)
involvement_dimension['InvolvementKey'] = range(1, len(involvement_dimension)+1)
involvement_dimension = involvement_dimension.rename(columns={'Bus Involvement': 'BusInvolvement',
                                                              'Heavy Rigid Truck Involvement': 'HeavyRigidTruckInvolvement',
                                                              'Articulated Truck Involvement': 'ArticulatedTruckInvolvement'})
involvement_dimension = involvement_dimension[['InvolvementKey','BusInvolvement','HeavyRigidTruckInvolvement','ArticulatedTruckInvolvement']]
involvement_dimension

,InvolvementKey,BusInvolvement,HeavyRigidTruckInvolvement,ArticulatedTruckInvolvement
0,1,No,No,No
1,2,No,No,Yes
2,3,Yes,No,Yes
3,4,No,Yes,No
4,5,No,Yes,Yes
5,6,Yes,No,No
6,7,Yes,Yes,No
7,8,No,Unknown_Involvement,No
8,9,No,Unknown_Involvement,Yes
9,10,Yes,Unknown_Involvement,No


### Crash Type Dimension

In [37]:
# crash_dimension = fatalities_clean[['Crash ID','Crash Type','Bus Involvement','Heavy Rigid Truck Involvement','Articulated Truck Involvement']].drop_duplicates().reset_index(drop=True)
# crash_dimension = pd.merge(crash_dimension,involvement_dimension,
#                             left_on=['Bus Involvement','Heavy Rigid Truck Involvement','Articulated Truck Involvement'],
#                             right_on=['BusInvolvement','HeavyRigidTruckInvolvement','ArticulatedTruckInvolvement'],
#                             how='left')
# crash_dimension = crash_dimension.rename(columns={'Crash ID': 'CrashID',
#                                                   'Crash Type': 'CrashType'})
# crash_dimension = crash_dimension[['CrashID','CrashType','InvolvementKey']]
# crash_dimension

In [38]:
crash_dimension = fatalities_clean[['Crash Type']].drop_duplicates().reset_index(drop=True)
crash_dimension['CrashTypeKey'] = np.where(crash_dimension['Crash Type']=='Single','S','M')
crash_dimension = crash_dimension.rename(columns={'Crash Type': 'CrashType'})
crash_dimension = crash_dimension[['CrashTypeKey','CrashType']]
crash_dimension

,CrashTypeKey,CrashType
0,S,Single
1,M,Multiple


### Location Dimension 

In [39]:
location_dimension = fatalities_clean[['State','National LGA Name 2021']].drop_duplicates().reset_index(drop=True)
location_dimension['LocationKey'] = range(1, len(location_dimension)+1)
location_dimension = location_dimension.rename(columns={'National LGA Name 2021': 'LGA'})
location_dimension = location_dimension[['LocationKey','State','LGA']]
location_dimension

,LocationKey,State,LGA
0,1,NSW,Wagga Wagga
1,2,NSW,Hawkesbury
2,3,Tas,Northern Midlands
3,4,NSW,Armidale Regional
4,5,Qld,Lockyer Valley
...,...,...,...
513,514,NSW,Unidentified_NSW
514,515,WA,Boyup Brook
515,516,WA,Cottesloe
516,517,WA,Leonora


In [40]:
lga_dimension = pd.read_excel('Population estimates by LGA, Significant Urban Area, Remoteness Area, Commonwealth Electoral Division and State Electoral Division, 2001 to 2023.xlsx',
                              sheet_name='Table 1', skiprows=6, usecols='A:Y', nrows=547)
lga_dimension.head()

,LGA code,Local Government Area,no.,no..1,no..2,no..3,no..4,no..5,no..6,no..7,...,no..13,no..14,no..15,no..16,no..17,no..18,no..19,no..20,no..21,no..22
0,10050,Albury,45265,45816,46180,46505,47004,47566,48140,48518,...,50990,51486,52171,53056,53922,54657,55466,56067,56665,57517
1,10180,Armidale,27906,27774,27610,27410,27350,27377,27468,27788,...,29015,29160,29310,29519,29631,29701,29600,29332,29361,29594
2,10250,Ballina,37856,38417,38870,39120,39305,39537,39824,40020,...,41881,42336,42993,43652,44385,44997,45663,46196,46849,47279
3,10300,Balranald,2751,2703,2661,2596,2545,2507,2473,2433,...,2376,2364,2330,2338,2308,2287,2257,2208,2210,2202
4,10470,Bathurst,35504,35831,36084,36245,36547,36916,37272,37904,...,41157,41694,42244,42583,42882,43207,43444,43674,44110,44612


In [41]:
lga_dimension = lga_dimension.rename(columns={'LGA code':'LGACode',
                                              'Local Government Area':'LGA'})
cols = lga_dimension.columns
rename_dict = {old: str(year) for old, year in zip(cols[2:], range(2001, 2024))}
lga_dimension = lga_dimension.rename(columns=rename_dict)

In [42]:
lga_dimension.tail()

,LGACode,LGA,2001,2002,2003,2004,2005,2006,2007,2008,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
542,74660,West Arnhem,6241,6198,6161,6182,6304,6452,6505,6679,...,7157,7060,6941,6943,6985,7059,7130,7182,7258,7407
543,74680,West Daly,2543,2612,2674,2752,2864,2986,3023,3151,...,3587,3598,3601,3536,3476,3427,3422,3422,3434,3426
544,79399,Unincorporated NT,7027,7072,7058,7128,7447,7664,7879,7995,...,7970,7112,7065,7034,7023,7102,7263,7420,7571,7713
545,89399,Unincorporated ACT,321538,324627,327357,328940,331399,335170,342644,348368,...,388799,395813,403104,415046,426081,435730,444903,452508,456915,466566
546,99399,Unincorp. Other Territories,542,464,441,428,413,386,370,370,...,361,367,2159,2243,2324,2382,2437,2530,2518,2516


In [43]:
lga_dimension[['LGACode']] = lga_dimension[['LGACode']].astype(str) #Convert code to str
lga_dimension

,LGACode,LGA,2001,2002,2003,2004,2005,2006,2007,2008,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,10050,Albury,45265,45816,46180,46505,47004,47566,48140,48518,...,50990,51486,52171,53056,53922,54657,55466,56067,56665,57517
1,10180,Armidale,27906,27774,27610,27410,27350,27377,27468,27788,...,29015,29160,29310,29519,29631,29701,29600,29332,29361,29594
2,10250,Ballina,37856,38417,38870,39120,39305,39537,39824,40020,...,41881,42336,42993,43652,44385,44997,45663,46196,46849,47279
3,10300,Balranald,2751,2703,2661,2596,2545,2507,2473,2433,...,2376,2364,2330,2338,2308,2287,2257,2208,2210,2202
4,10470,Bathurst,35504,35831,36084,36245,36547,36916,37272,37904,...,41157,41694,42244,42583,42882,43207,43444,43674,44110,44612
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
542,74660,West Arnhem,6241,6198,6161,6182,6304,6452,6505,6679,...,7157,7060,6941,6943,6985,7059,7130,7182,7258,7407
543,74680,West Daly,2543,2612,2674,2752,2864,2986,3023,3151,...,3587,3598,3601,3536,3476,3427,3422,3422,3434,3426
544,79399,Unincorporated NT,7027,7072,7058,7128,7447,7664,7879,7995,...,7970,7112,7065,7034,7023,7102,7263,7420,7571,7713
545,89399,Unincorporated ACT,321538,324627,327357,328940,331399,335170,342644,348368,...,388799,395813,403104,415046,426081,435730,444903,452508,456915,466566


In [44]:
location_dimension = pd.merge(location_dimension, lga_dimension, on='LGA', how='left')
location_dimension = location_dimension[['LocationKey','State','LGA','LGACode']]
location_dimension[location_dimension['LGACode'].isna()]

,LocationKey,State,LGA,LGACode
3,4,NSW,Armidale Regional,NaN
6,7,WA,Unidentified_WA,NaN
19,20,NSW,Mid-Western Regional,NaN
37,38,NSW,Central Coast,NaN
38,39,NSW,Tamworth Regional,NaN
87,88,NSW,Queanbeyan-Palerang Regional,NaN
88,89,NSW,Snowy Monaro Regional,NaN
105,106,NSW,Campbelltown,NaN
146,147,Vic,Moreland,NaN
163,164,NSW,Unincorporated,NaN


In [45]:
lga_dimension[lga_dimension['LGA'].str.contains(r'Armidale|Mid-Western|Tamworth|Queanbeyan-Palerang|Snowy Monaro|Bathurst|Dubbo',
                                                          na = False)]['LGA']

1                 Armidale
4                 Bathurst
36                   Dubbo
74             Mid-Western
94     Queanbeyan-Palerang
101           Snowy Monaro
106               Tamworth
Name: LGA, dtype: object

In [46]:
location_dimension[location_dimension['LGA'].str.contains(r'Armidale|Mid-Western|Tamworth|Queanbeyan-Palerang|Snowy Monaro|Bathurst|Dubbo',
                                                          na = False)]

,LocationKey,State,LGA,LGACode
3,4,NSW,Armidale Regional,NaN
19,20,NSW,Mid-Western Regional,NaN
38,39,NSW,Tamworth Regional,NaN
87,88,NSW,Queanbeyan-Palerang Regional,NaN
88,89,NSW,Snowy Monaro Regional,NaN
173,174,NSW,Bathurst Regional,NaN
180,181,NSW,Dubbo Regional,NaN


In [47]:
location_dimension[location_dimension['LGA'].str.contains('Regional',
                                                          na = False)]

,LocationKey,State,LGA,LGACode
3,4,NSW,Armidale Regional,NaN
19,20,NSW,Mid-Western Regional,NaN
38,39,NSW,Tamworth Regional,NaN
87,88,NSW,Queanbeyan-Palerang Regional,NaN
88,89,NSW,Snowy Monaro Regional,NaN
173,174,NSW,Bathurst Regional,NaN
180,181,NSW,Dubbo Regional,NaN


In [48]:
# Remove the ' Regional' from LGA names
location_dimension['LGA'] = location_dimension['LGA'].str.replace(' Regional', '', regex=False)
fatalities_clean['National LGA Name 2021'] = fatalities_clean['National LGA Name 2021'].str.replace(' Regional', '', regex=False)#Update main dataset

In [49]:
lga_dimension[lga_dimension['LGA'].str.contains('Unincorp', na = False)]['LGA']

128             Unincorporated NSW
208             Unincorporated Vic
357              Unincorporated SA
544              Unincorporated NT
545             Unincorporated ACT
546    Unincorp. Other Territories
Name: LGA, dtype: object

In [50]:
location_dimension[location_dimension['LGA'].str.contains('Unincorp', na = False)]

,LocationKey,State,LGA,LGACode
32,33,SA,Unincorporated SA,49399
82,83,ACT,Unincorporated ACT,89399
163,164,NSW,Unincorporated,NaN
301,302,Vic,Unincorporated Vic,29399
339,340,NT,Unincorporated NT,79399


In [51]:
location_dimension['LGA'] = np.where(location_dimension['LGA'] == 'Unincorporated', 
                                     'Unincorporated '+location_dimension['State'],
                                     location_dimension['LGA']) #Add the state to the unincorporated LGA names
location_dimension[location_dimension['LGA'].str.contains('Unincorp', na = False)]

,LocationKey,State,LGA,LGACode
32,33,SA,Unincorporated SA,49399
82,83,ACT,Unincorporated ACT,89399
163,164,NSW,Unincorporated NSW,NaN
301,302,Vic,Unincorporated Vic,29399
339,340,NT,Unincorporated NT,79399


In [52]:
fatalities_clean['National LGA Name 2021'] = np.where(fatalities_clean['National LGA Name 2021'] == 'Unincorporated', 
                                     'Unincorporated '+fatalities_clean['State'],
                                     fatalities_clean['National LGA Name 2021']) #Update main dataset

In [53]:
lga_dimension[lga_dimension['LGA'].str.contains(r'Central Coast|Campbelltown|Moreland|Bayside|Nambucca|Merri-bek|Anangu|Break',
                                                          na = False)]['LGA']

5                              Bayside (NSW)
21                        Campbelltown (NSW)
25                       Central Coast (NSW)
80                           Nambucca Valley
135                           Bayside (Vic.)
174                                Merri-bek
291    Anangu Pitjantjatjara Yankunytjatjara
296                        Campbelltown (SA)
497                              Break O`Day
500                     Central Coast (Tas.)
Name: LGA, dtype: object

In [54]:
location_dimension[location_dimension['LGA'].str.contains(r'Central Coast|Campbelltown|Moreland|Bayside|Nambucca|Anangu|Break',
                                                          na = False)]

,LocationKey,State,LGA,LGACode
29,30,NSW,Nambucca Valley,15700
37,38,NSW,Central Coast,NaN
105,106,NSW,Campbelltown,NaN
146,147,Vic,Moreland,NaN
194,195,SA,Campbelltown (SA),40910
198,199,Tas,Break O'Day,NaN
226,227,NSW,Bayside,NaN
311,312,Tas,Central Coast (Tas.),60810
416,417,Vic,Bayside (Vic.),20910
469,470,SA,Anangu Pitjantjatjara Yunkunytjatjara,NaN


Mismatched issue in the data are identified:

- Some LGA with the same name missing the State indication in their names
- Moreland was changed to Merri-bek: https://www.merri-bek.vic.gov.au/exploring-merri-bek/about-merri-bek/
- Nambucca Valley, Anangu, Break O`Day were recorded differently 

In [55]:
lga_missing_state = ['Bayside','Campbelltown','Central Coast']
location_dimension['LGA'] = np.where(location_dimension['LGA'].isin(lga_missing_state),
                                     location_dimension['LGA'] + ' ('+location_dimension['State']+')',
                                     location_dimension['LGA']) #Add the state to the LGA names

In [56]:
fatalities_clean['National LGA Name 2021'] = np.where(fatalities_clean['National LGA Name 2021'].isin(lga_missing_state),
                                     fatalities_clean['National LGA Name 2021'] + ' ('+fatalities_clean['State']+')',
                                     fatalities_clean['National LGA Name 2021']) #Update main dataset

In [57]:
replacements = {
    'Moreland': 'Merri-bek',
    'Anangu Pitjantjatjara Yunkunytjatjara': 'Anangu Pitjantjatjara Yankunytjatjara',
    "Break O'Day": 'Break O`Day',
    'Nambucca': 'Nambucca Valley',
}
location_dimension['LGA'] = location_dimension['LGA'].replace(replacements)

In [58]:
fatalities_clean['National LGA Name 2021'] = fatalities_clean['National LGA Name 2021'].replace(replacements) #Update main dataset

In [59]:
location_dimension = pd.merge(location_dimension[['State','LGA']].drop_duplicates().reset_index(drop=True), 
                              lga_dimension, on='LGA', how='left')
location_dimension['LocationKey'] = range(1, len(location_dimension)+1)
location_dimension = location_dimension[['LocationKey','State','LGA','LGACode']]
location_dimension[location_dimension['LGACode'].isna()]

,LocationKey,State,LGA,LGACode
6,7,WA,Unidentified_WA,NaN
189,190,NT,Unidentified_NT,NaN
307,308,Vic,Unidentified_Vic,NaN
323,324,ACT,Unidentified_ACT,NaN
450,451,Tas,Unidentified_Tas,NaN
505,506,Qld,Unidentified_Qld,NaN
512,513,NSW,Unidentified_NSW,NaN
516,517,SA,Unidentified_SA,NaN


In [60]:
location_dimension

,LocationKey,State,LGA,LGACode
0,1,NSW,Wagga Wagga,17750
1,2,NSW,Hawkesbury,13800
2,3,Tas,Northern Midlands,64610
3,4,NSW,Armidale,10180
4,5,Qld,Lockyer Valley,34580
...,...,...,...,...
512,513,NSW,Unidentified_NSW,NaN
513,514,WA,Boyup Brook,50770
514,515,WA,Cottesloe,52170
515,516,WA,Leonora,55040


### Population Dimension

In [61]:
population_dimension = lga_dimension.melt(id_vars=['LGACode','LGA'], var_name='Year', value_name='Population')

# Convert 'Year' to integer
population_dimension['Year'] = population_dimension['Year'].astype(int)
population_dimension['LGACode'] = population_dimension['LGACode'].astype(str)
population_dimension['PopulationKey'] = range(1, len(population_dimension)+1)
population_dimension = pd.merge(population_dimension, location_dimension[['LocationKey','LGACode']], on='LGACode', how='left')
population_dimension = population_dimension[['PopulationKey','LocationKey','LGACode','LGA','Year','Population']]
population_dimension[population_dimension['LocationKey'].isna()][['LGA','LGACode']].drop_duplicates()

,LGA,LGACode
189,Queenscliffe,26080
209,Aurukun,30250
214,Blackall Tambo,30760
215,Boulia,30900
226,Cherbourg,32330
230,Diamantina,32750
244,Kowanyama,34420
252,Mapoon,34830
259,Napranum,35670
264,Paroo,35800


In [62]:
new_lgas = population_dimension[population_dimension['LocationKey'].isna()][['LGA','LGACode']].drop_duplicates()
# Append new rows
location_dimension = pd.concat([location_dimension, new_lgas], ignore_index=True)
location_dimension['LocationKey'] = range(1, len(location_dimension)+1)
location_dimension = location_dimension[['LocationKey','State','LGA','LGACode']]
location_dimension['LGACode'] = location_dimension['LGACode'].astype(str) #Convert code to int instead of float
location_dimension.tail(10)

,LocationKey,State,LGA,LGACode
545,546,NaN,Tammin,58190
546,547,NaN,Three Springs,58260
547,548,NaN,Upper Gascoyne,58470
548,549,NaN,Wiluna,59250
549,550,NaN,Wyalkatchem,59330
550,551,NaN,Yalgoo,59350
551,552,NaN,Belyuen,70540
552,553,NaN,Darwin Waterfront Precinct,71150
553,554,NaN,Tiwi Islands,74050
554,555,NaN,Unincorp. Other Territories,99399


In [63]:
duplicates_exist = location_dimension.duplicated().any()
print(f"Any Duplicates? {duplicates_exist}")

Any Duplicates? False


In [64]:
population_dimension = pd.merge(population_dimension[['PopulationKey','LGACode','LGA','Year','Population']], location_dimension[['LocationKey','LGACode']], on='LGACode', how='left')
population_dimension = population_dimension[['PopulationKey','LocationKey','LGACode','LGA','Year','Population']]
population_dimension[population_dimension['LocationKey'].isna()][['LGA','LGACode']].drop_duplicates()

,LGA,LGACode


The data are all matched

In [65]:
population_dimension = population_dimension[['PopulationKey','LocationKey','Year','Population']]
population_dimension

,PopulationKey,LocationKey,Year,Population
0,1,132,2001,45265
1,2,4,2001,27906
2,3,56,2001,37856
3,4,46,2001,2751
4,5,174,2001,35504
...,...,...,...,...
12576,12577,352,2023,7407
12577,12578,498,2023,3426
12578,12579,340,2023,7713
12579,12580,83,2023,466566


### Holiday Dimension

In [66]:
holiday_dimension = pd.DataFrame({
    'HolidayKey': range(1,4),
    'Holiday':['No Holiday','Christmas','Easter'],
})

In [67]:
holiday_dimension

,HolidayKey,Holiday
0,1,No Holiday
1,2,Christmas
2,3,Easter


### Export Dimension to CSV file

In [68]:
location_dimension.to_csv("location_dimension.csv", index = False)
time_dimension.to_csv("time_dimension.csv", index = False)
date_dimension.to_csv("date_dimension.csv", index = False)
road_user_dimension.to_csv("road_user_dimension.csv", index = False)
gender_dimension.to_csv("gender_dimension.csv", index = False)
age_dimension.to_csv("age_dimension.csv", index = False)
road_dimension.to_csv("road_dimension.csv", index = False)
speed_dimension.to_csv("speed_dimension.csv", index = False)
population_dimension.to_csv("population_dimension.csv", index = False)
involvement_dimension.to_csv("involvement_dimension.csv", index = False)
crash_dimension.to_csv("crash_dimension.csv", index = False)
holiday_dimension.to_csv("holiday_dimension.csv", index = False)

# Fact table extraction

In [69]:
fact_factalities = pd.merge(fatalities_clean, 
                            date_dimension,
                            on=['Dayweek','Month','Year'],
                            how='left')
fact_fatalities = pd.merge(fact_factalities, 
                           time_dimension,
                           on = 'Time',
                           how='left')
fact_fatalities = pd.merge(fact_fatalities, 
                            road_user_dimension,
                            left_on=['Road User'],
                            right_on=['RoadUser'],
                            how='left')
fact_fatalities = pd.merge(fact_fatalities,
                           gender_dimension,
                           left_on = 'Gender',
                           right_on = 'Gender',
                           how='left')
fact_fatalities = pd.merge(fact_fatalities,
                           age_dimension,
                           on = 'Age',
                           how='left')
fact_fatalities = pd.merge(fact_fatalities,
                           speed_dimension,
                           left_on = 'Speed Limit',
                           right_on = 'SpeedLimit',
                           how='left')
fact_fatalities = pd.merge(fact_fatalities,
                           road_dimension,
                           left_on = 'National Road Type',
                           right_on = 'RoadType',
                           how='left')
fact_fatalities = pd.merge(fact_fatalities,
                           involvement_dimension,
                           left_on = ['Bus Involvement','Heavy Rigid Truck Involvement','Articulated Truck Involvement'],
                           right_on = ['BusInvolvement','HeavyRigidTruckInvolvement','ArticulatedTruckInvolvement'],
                           how='left')
fact_fatalities = pd.merge(fact_fatalities,
                           crash_dimension,
                           left_on = 'Crash Type',
                           right_on = 'CrashType',
                           how='left')
fact_fatalities = pd.merge(fact_fatalities,
                           location_dimension,
                           left_on = ['State','National LGA Name 2021'],
                           right_on = ['State','LGA'],
                           how='left')

In [70]:
conditions = [
    fact_fatalities['Christmas Period'] == 'Yes',
    fact_fatalities['Easter Period'] == 'Yes'
]
values = [2,3]
fact_fatalities['HolidayKey'] = np.select(conditions, values, default=1)
fact_fatalities['fat_id'] = range(1, len(fact_fatalities)+1)
fact_fatalities= fact_fatalities[['fat_id','Crash ID', 'DateKey', 'TimeKey','HolidayKey','RoadUserKey',
       'GenderKey', 'RoadKey', 'InvolvementKey','SpeedLimit','Age',
       'CrashTypeKey', 'LocationKey']]
fact_fatalities['Age'] = fact_fatalities['Age'].astype('Int64') #Convert age to int

In [71]:
fact_fatalities = fact_fatalities.rename(columns={'Crash ID': 'CrashKey'})

In [72]:
fact_fatalities.isna().sum()

fat_id            0
CrashKey          0
DateKey           0
TimeKey           0
HolidayKey        0
RoadUserKey       0
GenderKey         0
RoadKey           0
InvolvementKey    0
SpeedLimit        0
Age               0
CrashTypeKey      0
LocationKey       0
dtype: int64

In [73]:
assert len(fact_fatalities) == len(fatalities_clean) #checking if the merged process creating duplicates
fact_fatalities.to_csv('fact_fatalities.csv', index=False) #Export the fact table to csv

In [74]:
fatalities_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 55183 entries, 0 to 56872
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Crash ID                       55183 non-null  int64  
 1   State                          55183 non-null  object 
 2   Month                          55183 non-null  int64  
 3   Year                           55183 non-null  int64  
 4   Dayweek                        55183 non-null  object 
 5   Time                           55183 non-null  object 
 6   Crash Type                     55183 non-null  object 
 7   Bus Involvement                55183 non-null  object 
 8   Heavy Rigid Truck Involvement  55183 non-null  object 
 9   Articulated Truck Involvement  55183 non-null  object 
 10  Speed Limit                    55183 non-null  int32  
 11  Road User                      55183 non-null  object 
 12  Gender                         55183 non-null  obje